# Using Ray with TimeCopilot

In [2]:

import nest_asyncio

nest_asyncio.apply()

from timecopilot import TimeCopilotForecaster

import ray 

from timecopilot.models import SeasonalNaive
from timecopilot.models.foundation.chronos import Chronos

2026-03-23 15:35:03,721	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.


## Create the dataframe

In [ ]:
df = ray.data.read_csv("https://timecopilot.s3.amazonaws.com/public/data/events_pageviews.csv")

2026-03-23 15:35:05,235	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-03-23 15:35:07,522	INFO worker.py:1927 -- Started a local Ray instance.


                                                                                                                              
                                                                                                                                                          

                                                                                                                                                    


                                                   



                                                




                                                





Running Dataset: dataset_10_0. Active & requested resources: 3/14 CPU, 768.0MB/1.0GB object store: : 0.00 row [00:05, ? row/s]                         
















(MapBatches(_udf) pid=21501)  See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.


                                                                                                                              
                                                                                                                                                          

                                                                                                                                                    


                                                   



                                                




                                                





                                                                                                                                     (MapBatches(_udf) pid=21491) `torch_dtype` is deprecated! Use `dtype` instead!
Running Dataset: dataset_10_0. Active & requested resources: 3/14 CPU, 768.0MB/1.0GB object store: : 0.00 row [00:07, ? row/s]














                                                   

(MapBatches(_udf) pid=21497)  See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs. [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)


                                                                                                                             
                                                                                                                                                          

                                                                                                                                                   


                                                   



                                                




                                                





                                                                                                                              (MapBatches(_udf) pid=21494) `torch_dtype` is deprecated! Use `dtype` instead! [repeated 5x across cluster]
Running Dataset: dataset_10_0. Active & requested resources: 11/14 CPU, 9.1KB/1.0GB object store: : 0.00 row [00:15, ? row/s]














                                

In [4]:
df.show(5)

2026-03-23 15:35:08,973	INFO dataset.py:3055 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-03-23 15:35:08,978	INFO logging.py:295 -- Registered dataset logger for dataset dataset_1_0
2026-03-23 15:35:08,997	INFO streaming_executor.py:117 -- Starting execution of Dataset dataset_1_0. Full logs are in /tmp/ray/session_2026-03-23_15-35-05_459775_21356/logs/ray-data
2026-03-23 15:35:08,997	INFO streaming_executor.py:118 -- Execution plan of Dataset dataset_1_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV] -> LimitOperator[limit=5]
Running 0: 0.00 row [00:00, ? row/s]
2026-03-23 15:35:09,013	WARNING resource_manager.py:130 -- ⚠️  Ray's object store is configured to use only 6.9% of available memory (2.1GB out of 31.3GB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.ini

{'unique_id': 'Oktoberfest', 'ds': datetime.date(2020, 1, 31), 'y': 25376}
{'unique_id': 'Oktoberfest', 'ds': datetime.date(2020, 2, 29), 'y': 28470}
{'unique_id': 'Oktoberfest', 'ds': datetime.date(2020, 3, 31), 'y': 23816}
{'unique_id': 'Oktoberfest', 'ds': datetime.date(2020, 4, 30), 'y': 46186}
{'unique_id': 'Oktoberfest', 'ds': datetime.date(2020, 5, 31), 'y': 31213}


## Create a TimeCopilotForecaster

In [5]:
tcf = TimeCopilotForecaster(
    models=[
        SeasonalNaive(),
        Chronos("autogluon/chronos-2-small")
    ]
)

## Create a forecast

In [6]:
result = tcf.forecast(
    df=df,
    h=12
)

2026-03-23 15:35:18,354	INFO logging.py:295 -- Registered dataset logger for dataset dataset_2_0
2026-03-23 15:35:18,356	INFO streaming_executor.py:117 -- Starting execution of Dataset dataset_2_0. Full logs are in /tmp/ray/session_2026-03-23_15-35-05_459775_21356/logs/ray-data
2026-03-23 15:35:18,356	INFO streaming_executor.py:118 -- Execution plan of Dataset dataset_2_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV]
Running 0: 0.00 row [00:00, ? row/s]2026-03-23 15:35:18,502	INFO streaming_executor.py:231 -- ✔️  Dataset dataset_2_0 execution finished in 0.15 seconds
                                                                                                               
✔️  Dataset dataset_2_0 execution finished in 0.15 seconds: 100%|██████████| 408/408 [00:00<00:00, 2.76k row/s]

- ReadCSV->SplitBlocks(28): Tasks: 0; Actors: 0; Queued blocks: 0; Resources: 0.0 CPU, 7.2KB object store: : 408 row [00:00, 2.75k row/s]
2026-03-23 15:35:18,551	INFO logging.py:295 -- Regist

## View Forecast Results

In [10]:
from IPython.display import HTML
display(HTML(result.to_pandas().to_html()))

,unique_id,ds,SeasonalNaive,Chronos
0,Oktoberfest,2025-09-30,181474.0,205939.90625
1,Oktoberfest,2025-10-31,142094.0,161174.0625
2,Oktoberfest,2025-11-30,36200.0,35124.382812
3,Oktoberfest,2025-12-31,26387.0,30676.568359
4,Oktoberfest,2026-01-31,27944.0,31133.820312
5,Oktoberfest,2026-02-28,25764.0,30749.806641
6,Oktoberfest,2026-03-31,30708.0,31413.792969
7,Oktoberfest,2026-04-30,31258.0,31530.412109
8,Oktoberfest,2026-05-31,36179.0,32509.914062
9,Oktoberfest,2026-06-30,33992.0,32520.060547
